In [ ]:
 !pip install nltk
 import nltk
 nltk.download('all')

# 안돌려도 됨
# 코드 돌리는

[nltk_data] Downloading collection 'all'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to /root/nltk_data...
[nltk_data]    |   Package abc is already up-to-date!
[nltk_data]    | Downloading package alpino to /root/nltk_data...
[nltk_data]    |   Package alpino is already up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger is already up-
[nltk_data]    |       to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_eng to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger_eng is already
[nltk_data]    |       up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_ru to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger_ru is already
[nltk_data]    |       up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_r

True

In [ ]:
"""
VADER 기반 프롬프트 감성 검증기
=================================
입력 CSV: target_level, prompt
출력 CSV: target_level, prompt, vader_score, score_gap, accepted, reject_reason
"""

import csv, os
from nltk.sentiment.vader import SentimentIntensityAnalyzer

In [ ]:
# ── 설정 ──────────────────────────────────────────────────────────────
INPUT_FILE = "/content/prompts_template_final.csv"   # 예) "/Users/bluestar/project/prompts.csv"

SCORE_TOLERANCE = 0.35

# ── 출력 파일 자동 생성 (입력 파일명_validated.csv) ───────────────────
base, ext = os.path.splitext(INPUT_FILE)
OUTPUT_FILE = f"{base}_validated{ext}"

# ── 초기화 ────────────────────────────────────────────────────────────
vader = SentimentIntensityAnalyzer()

# ── 함수 ──────────────────────────────────────────────────────────────
def get_vader_score(text: str) -> float:
    return round(vader.polarity_scores(text)["compound"], 4)

def validate(target: float, v_score: float) -> tuple[bool, str]:
    gap = round(abs(v_score - target), 4)
    if gap <= SCORE_TOLERANCE:
        return True, "OK"
    else:
        return False, f"점수 오차 과대({gap:.4f} > {SCORE_TOLERANCE})"

# ── 메인 처리 ─────────────────────────────────────────────────────────
results = []

with open(INPUT_FILE, newline="", encoding="utf-8-sig") as f:
    for row in csv.DictReader(f):
        target = float(row["target_level"])
        prompt = row["prompt"].strip()

        if not prompt:
            continue

        v_score          = get_vader_score(prompt)
        gap              = round(abs(v_score - target), 4)
        accepted, reason = validate(target, v_score)

        results.append({
            "target_level" : target,
            "prompt"       : prompt,
            "vader_score"  : v_score,
            "score_gap"    : gap,
            "accepted"     : "TRUE" if accepted else "FALSE",
            "reject_reason": reason,
        })

# ── CSV 출력 ──────────────────────────────────────────────────────────
out_fields = ["target_level", "prompt",
              "vader_score", "score_gap", "accepted", "reject_reason"]

with open(OUTPUT_FILE, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(f, fieldnames=out_fields)
    writer.writeheader()
    writer.writerows(results)

# ── 콘솔 요약 ─────────────────────────────────────────────────────────
accepted_n = sum(1 for r in results if r["accepted"] == "TRUE")

print(f"\n{'#':<5} {'target':>7} {'vader':>8} {'gap':>7}  결과")
print("─" * 60)
for i, r in enumerate(results, 1):
    tag = "채택" if r["accepted"] == "TRUE" else f"{r['reject_reason']}"
    print(f"{i:<5} {r['target_level']:>7} {r['vader_score']:>8} {r['score_gap']:>7}  {tag}")

print(f"\n총 {len(results)}개 | 채택: {accepted_n}개 | 탈락: {len(results)-accepted_n}개")
print(f"출력 파일: {OUTPUT_FILE}")


#      target    vader     gap  결과
────────────────────────────────────────────────────────────
1        -1.0   -0.763   0.237  채택
2        -1.0  -0.8211  0.1789  채택
3        -1.0  -0.8008  0.1992  채택
4        -1.0  -0.8442  0.1558  채택
5        -1.0  -0.8658  0.1342  채택
6        -1.0  -0.9371  0.0629  채택
7        -1.0  -0.8971  0.1029  채택
8        -1.0  -0.9348  0.0652  채택
9        -1.0  -0.9226  0.0774  채택
10       -1.0  -0.6801  0.3199  채택
11       -0.5  -0.4417  0.0583  채택
12       -0.5  -0.6534  0.1534  채택
13       -0.5  -0.5888  0.0888  채택
14       -0.5  -0.4988  0.0012  채택
15       -0.5  -0.7269  0.2269  채택
16       -0.5  -0.4417  0.0583  채택
17       -0.5  -0.2516  0.2484  채택
18       -0.5  -0.2516  0.2484  채택
19       -0.5  -0.5423  0.0423  채택
20       -0.5  -0.5153  0.0153  채택
21        0.0      0.0     0.0  채택
22        0.0      0.0     0.0  채택
23        0.0      0.0     0.0  채택
24        0.0      0.0     0.0  채택
25        0.0      0.0     0.0  채택
26        0.0      0.0     0

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
